# Notebook 01: Data Preparation

**Paper 3** — Physics-Informed Neural Networks for Cloud Droplet Profile Retrieval  
Andrew J. Buggee, LASP / CU Boulder

This notebook:
1. Generates random cloud state vectors (r_top, r_bot, τ_c) with adiabatic profiles
2. Runs libRadtran/UVSPEC to compute TOA reflectance spectra for each state
3. Packages everything into an HDF5 file for training
4. Generates a smaller synthetic test dataset for quick iteration

**Note:** The full libRadtran dataset (~500k samples) should be generated on Alpine
using SLURM job arrays. This notebook demonstrates the pipeline on a small subset.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add project root to path so we can import our modules
sys.path.insert(0, str(Path('.').resolve().parent))
from src.data import (
    adiabatic_profile, generate_random_profiles,
    MODIS_WAVELENGTHS, DEFAULT_N_LEVELS
)

# Reproducibility
rng = np.random.default_rng(42)

print(f"MODIS wavelengths (nm): {MODIS_WAVELENGTHS}")
print(f"Number of vertical levels: {DEFAULT_N_LEVELS}")

## 1. Generate Random Cloud State Vectors

We sample (r_top, r_bot, τ_c) randomly, compute adiabatic profiles,
and optionally add sub-adiabatic perturbations to train a network
that can handle non-ideal clouds.

In [ ]:
# Generate a small demo dataset (increase n_samples for real training)
N_SAMPLES = 1000  # Use ~500,000 for actual training on Alpine
N_LEVELS = 10

data = generate_random_profiles(
    n_samples=N_SAMPLES,
    n_levels=N_LEVELS,
    r_top_range=(4.0, 20.0),   # μm
    r_bot_range=(1.0, None),    # max = r_top (adiabatic constraint)
    tau_range=(3.0, 30.0),
    include_subadiabatic=True,
    rng=rng,
)

print(f"Profiles shape: {data['profiles'].shape}")
print(f"r_top range: [{data['r_top'].min():.2f}, {data['r_top'].max():.2f}] μm")
print(f"r_bot range: [{data['r_bot'].min():.2f}, {data['r_bot'].max():.2f}] μm")
print(f"τ_c range: [{data['tau_c'].min():.2f}, {data['tau_c'].max():.2f}]")

In [ ]:
# Visualize some example profiles
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Plot 20 random profiles
z_norm = np.linspace(0, 1, N_LEVELS)  # 0=top, 1=base (optical depth convention)
idx = rng.choice(N_SAMPLES, size=20, replace=False)

ax = axes[0]
for i in idx:
    ax.plot(data['profiles'][i], z_norm, alpha=0.5, linewidth=1)
ax.set_xlabel('Effective Radius (μm)', fontsize=12)
ax.set_ylabel('Normalized depth (0=top, 1=base)', fontsize=12)
ax.set_title('Sample Profiles', fontsize=13)
ax.invert_yaxis()

# Distribution of r_top
ax = axes[1]
ax.hist(data['r_top'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('r_top (μm)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Cloud Top Radius Distribution', fontsize=13)

# Distribution of τ_c
ax = axes[2]
ax.hist(data['tau_c'], bins=40, color='coral', edgecolor='white', alpha=0.8)
ax.set_xlabel('τ_c', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Optical Depth Distribution', fontsize=13)

plt.tight_layout()
plt.show()

## 2. Generate Reflectance Spectra with libRadtran

This section calls libRadtran/UVSPEC to compute TOA reflectances.

**For the full dataset on Alpine:** Use a SLURM job array where each task
computes reflectances for a batch of ~1000 state vectors. Your existing
MATLAB+libRadtran pipeline from Papers 1 & 2 can be adapted, or you
can call libRadtran from Python using subprocess.

Below is a placeholder that generates synthetic reflectances for
development and testing. Replace with actual libRadtran calls.

In [ ]:
def generate_synthetic_reflectances(profiles, tau_c, wavelengths, rng):
    """
    PLACEHOLDER: Generate synthetic reflectances for development.
    
    Replace this with actual libRadtran calls for real training data.
    
    This simplified model captures the key physics:
    - Visible reflectance increases with optical depth
    - SWIR reflectance decreases with droplet size
    - Noise added to simulate measurement uncertainty
    """
    n_samples = profiles.shape[0]
    n_wav = len(wavelengths)
    
    reflectances = np.zeros((n_samples, n_wav), dtype=np.float32)
    
    for i in range(n_samples):
        r_top = profiles[i, 0]   # cloud top radius
        r_bot = profiles[i, -1]  # cloud base radius
        tc = tau_c[i]
        
        for j, lam in enumerate(wavelengths):
            # Simple model: R depends on tau_c at visible, on r_e at SWIR
            if lam < 1000:  # Visible/near-IR
                # Reflectance roughly proportional to tau for thick clouds
                reflectances[i, j] = 0.3 + 0.02 * tc - 0.001 * r_top
            else:  # SWIR
                # Reflectance decreases with increasing absorption (larger r_e)
                absorption_factor = (lam / 1000) * 0.02 * r_top
                reflectances[i, j] = 0.3 + 0.015 * tc - absorption_factor
        
        # Add measurement noise (~2% relative)
        noise = rng.normal(0, 0.02, size=n_wav) * reflectances[i]
        reflectances[i] += noise
    
    # Clip to physical range
    reflectances = np.clip(reflectances, 0.01, 0.95)
    
    return reflectances


# Generate synthetic reflectances for development
reflectances = generate_synthetic_reflectances(
    data['profiles'], data['tau_c'], MODIS_WAVELENGTHS, rng
)

print(f"Reflectances shape: {reflectances.shape}")
print(f"Reflectance range: [{reflectances.min():.4f}, {reflectances.max():.4f}]")

In [ ]:
# TODO: Replace the above with actual libRadtran calls.
# Here is a template for calling libRadtran from Python:
#
# import subprocess
# 
# def run_libradtran(input_file, output_file, libradtran_path):
#     """Run libRadtran UVSPEC for a single input file."""
#     cmd = f"{libradtran_path}/bin/uvspec < {input_file} > {output_file}"
#     result = subprocess.run(cmd, shell=True, capture_output=True)
#     if result.returncode != 0:
#         raise RuntimeError(f"libRadtran failed: {result.stderr.decode()}")
#     return np.loadtxt(output_file)
#
# Your existing MATLAB wrapper generates UVSPEC input files.
# You can either:
#   (a) Port that to Python (recommended for this project)
#   (b) Call MATLAB from Python to generate input files, then run UVSPEC

## 3. Generate Geometry Inputs

The retrieval also needs solar-viewing geometry and surface wind speed.

In [ ]:
# Random geometry for training data
# These ranges cover typical MODIS observation conditions
sza = rng.uniform(10, 70, size=N_SAMPLES).astype(np.float32)       # Solar zenith angle
vza = rng.uniform(0, 55, size=N_SAMPLES).astype(np.float32)        # Viewing zenith angle
phi = rng.uniform(0, 180, size=N_SAMPLES).astype(np.float32)       # Relative azimuth
wind_speed = rng.uniform(1, 12, size=N_SAMPLES).astype(np.float32) # Surface wind speed (m/s)

print(f"SZA range: [{sza.min():.1f}, {sza.max():.1f}] degrees")
print(f"VZA range: [{vza.min():.1f}, {vza.max():.1f}] degrees")

## 4. Save Training Data to HDF5

In [ ]:
output_path = Path('../data')
output_path.mkdir(exist_ok=True)
h5_file = output_path / 'libradtran_training_modis7.h5'

with h5py.File(h5_file, 'w') as f:
    # Reflectances: input to the retrieval network
    f.create_dataset('reflectances', data=reflectances, compression='gzip')
    
    # Profiles: target output (r_e at N levels, top to base)
    f.create_dataset('profiles', data=data['profiles'], compression='gzip')
    
    # Optical depth: target output
    f.create_dataset('tau_c', data=data['tau_c'])
    
    # Geometry: additional inputs
    f.create_dataset('sza', data=sza)
    f.create_dataset('vza', data=vza)
    f.create_dataset('phi', data=phi)
    f.create_dataset('wind_speed', data=wind_speed)
    
    # Metadata
    f.create_dataset('wavelengths', data=MODIS_WAVELENGTHS)
    
    # Attributes
    f.attrs['n_samples'] = N_SAMPLES
    f.attrs['n_levels'] = N_LEVELS
    f.attrs['n_wavelengths'] = len(MODIS_WAVELENGTHS)
    f.attrs['description'] = 'Training data for cloud droplet profile retrieval (Paper 3)'
    f.attrs['forward_model'] = 'SYNTHETIC (replace with libRadtran)'

print(f"Saved training data to {h5_file}")
print(f"File size: {h5_file.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Verify we can read it back
with h5py.File(h5_file, 'r') as f:
    print("HDF5 contents:")
    for key in f.keys():
        ds = f[key]
        print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")
    print(f"\nAttributes:")
    for key, val in f.attrs.items():
        print(f"  {key}: {val}")